Download ERA5-Land climate data

#### set the internet proxy, sometime required. 

In [19]:
import os
import urllib.request
os.environ["http_proxy"] = "http://127.0.0.1:7993"
os.environ["https_proxy"] = "http://127.0.0.1:7993"
# 刷新 urllib 的代理设置
urllib.request.install_opener(urllib.request.build_opener())
with urllib.request.urlopen(
    "https://accounts.google.com", timeout=20
) as response:
    print("连接成功:", response.status) 


连接成功: 200


In [20]:
import ee  
import geemap 
 

In [21]:
# Initialize the Earth Engine module.
# Change your own Google Cloud project ID
MY_PROJECT_ID = 'earth-engine-auth-project'     # your-google-cloud-project-id  
try:
    ee.Initialize(project=MY_PROJECT_ID)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)
print("GEE Initialized successfully with project ID:", MY_PROJECT_ID) 


GEE Initialized successfully with project ID: earth-engine-auth-project


In [22]:
# Area of interest and year
region = ee.Geometry.Rectangle([65, 24, 107, 48], 'EPSG:4326', False)  ## region
year = '2000'
# ERA5-Land Monthly Aggregated   
era5_land = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
         .filter(ee.Filter.date(f'{year}-01-01', f'{year}-12-31'))
print(era5_land.size().getInfo())    



12


In [23]:
# 1. Temperature processing (2m temperature, Kelvin K to Celsius °C)
imgcol_t2m = era5_land.select(['temperature_2m'])
t2m_yearly_mean = imgcol_t2m.mean().clip(region).subtract(273.15)

# 2. Precipitation processing (cumulative precipitation, meters m to millimeters mm)
imgcol_tp = era5_land.select(['total_precipitation_sum'])
tp_yearly_sum = imgcol_tp.sum().clip(region).multiply(1000)

# 3. Total evaporation processing (cumulative total evaporation, meters m to millimeters mm)
imgcol_te = era5_land.select(['total_evaporation_sum'])
te_yearly_sum = imgcol_te.sum().clip(region).multiply(1000) 


In [24]:
# visualization parameters
visual = {
    'min': 0.0,
    # 'max': 40,  ## for temperature 
    'max':6000,
    'palette': [
        '000080', '0000d9', '4000ff', '8000ff', '0080ff', '00ffff',
        '00ff80', '80ff00', 'daff00', 'ffff00', 'fff500', 'ffda00',
        'ffb000', 'ffa400', 'ff4f00', 'ff2500', 'ff0a00', 'ff00ff',
    ]
}

# Area of interest outline
empty = ee.Image().byte()
scene_outline = empty.paint(
    featureCollection=region,
    color=1,
    width=3
)

In [25]:
# Visualize the results using geemap
Map = geemap.Map()
Map.centerObject(region, 4)
# Map.addLayer(t2m_yearly_mean, visual, 'total tempreture')
Map.addLayer(tp_yearly_sum, visual, 'total precipitation')
# Map.addLayer(te_yearly_sum, visual, 'total evaporation')
Map.addLayer(scene_outline, {'palette': 'FF0000'}, 'training region')

# Show the map in Jupyter Notebook
Map


Map(center=[35.998389099112636, 86], controls=(WidgetControl(options=['position', 'transparent_bg'], position=…

In [26]:
# ==============================================================
# 10. 导出到 Google Drive
# ==============================================================
# projection = dset.first().projection().getInfo()
# task = ee.batch.Export.image.toDrive(
#     image=te_yearly_sum,
#     description=f'era5_land_yearly_te_{year}',
#     folder='tmp',
#     scale=11132,
#     crs=projection['crs'],
#     crsTransform=projection['transform'],
#     fileFormat='GeoTIFF',
#     region=region
# )
# task.start()
# print(f"导出任务已提交，Task ID: {task.id}") 

